# Fourier basis — wavelength band

Supports Supplementary Note 1: how to choose the wavelength band
$[L_{\min}, L_{\max}]$. It is tested on the synthetic recumbent fold of
`01_recumbent_fold` (2-D, the truth known), with everything else exactly as
its single fit (`01b`, through `results/synthetic/fit_settings.json`).

One sweep: $L_{\min}$ and $L_{\max}$ on a grid, `N_SEEDS` Fourier seeds per band.

Each fit is scored as in `01d`, where the drillholes decide the geometry
(`recumbent_inputs.score_against_truth`): the median contact distance per
horizon, the IoU per unit, and the number of interior extrema. The
**displacement spectrum** (`diagnostics.displacement_spectrum`) says which
wavelengths carry the fitted map: each mode's share of the map's rms
displacement, summed per octave of wavelength.

Writes `results/band_sweep.npz`, read by
`s01d_fourier_basis_figures`. Runs in `curlew_dev`.

## Setup

In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import torch

sys.path.insert(0, str(Path("..").resolve()))                      # publication/: figstyle, diagnostics
sys.path.insert(0, str(Path("../01_recumbent_fold").resolve()))    # recumbent_inputs, synthetic_fold
import diagnostics
import recumbent_inputs

import curlew
from curlew.core import CSet, HSet
from curlew.geometry import Grid
from curlew.geology import restore
from curlew.kinematics.euler import euler_velocity

curlew.device = "cuda" if torch.cuda.is_available() else "cpu"
curlew.dtype = torch.float32
curlew.default_dim = 2

FOLD_DIR = Path("../01_recumbent_fold")
DATA_DIR = FOLD_DIR / "data"
P = json.loads((FOLD_DIR / "results/synthetic/fit_settings.json").read_text())   # written by 01b
RES_DIR = Path("./results")
RES_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(P, indent=1))

## Data, grid and truth

As in `01d`.

In [2]:
section = recumbent_inputs.load_section("synthetic", DATA_DIR)
horizon_names, horizons = section["horizon_names"], section["horizons"]
gp, gv, origin = section["gp"], section["gv"], section["origin"]
anchor = torch.zeros(2, dtype=curlew.dtype, device=curlew.device)
C = CSet(gp=gp, gv=gv, eq=horizons, iq=(P["iq_samples"], [(horizons[0], horizons[1], "<")]))

G = Grid((1200, 1200), step=(5.0, 5.0), center=np.zeros(2), sampleArgs=dict(N=128))
domaxes = G.tformed_axes
grd = torch.tensor(G.coords(), dtype=curlew.dtype, device=curlew.device)
unit_true = recumbent_inputs.true_units(domaxes[0], domaxes[1], origin, DATA_DIR)
reference = recumbent_inputs.truth_reference(origin, DATA_DIR)
X, Z = np.meshgrid(domaxes[0], domaxes[1], indexing="ij")
spectrum_points = np.c_[X.ravel(), Z.ravel()][::16]            # every 16th grid node is plenty for the split
print(f"section {domaxes[0][-1] - domaxes[0][0]:.0f} m wide; scored within x {np.round(reference['window'])} m")

section 1195 m wide; scored within x [-503.  430.] m


## One fit

`fit(band, seed)` builds the LDI event of `01d` with the band changed, fits
it, and scores it. The weight exponent stays at $\nu = 2$, as in every fit of
the paper. (Under gradient descent a different $\nu$ is the same as a
per-mode learning rate scaled by $\lVert\boldsymbol\omega\rVert^{2\nu-4}$, so a
comparison of $\nu$ at one learning rate would mostly compare step sizes.)

In [ ]:
OCTAVES = 2.0 ** np.arange(-5, 6)        # wavelength bin edges, as multiples of the section width
WIDTH = float(domaxes[0][-1] - domaxes[0][0])


def make_ldi(band, seed):
    velocity = euler_velocity(
        2, n_features=P["n_features"], n_directions=P["n_directions"], seed=int(seed),
        length_scale_range=tuple(band), stream="alpha", activation="cubic",
        energy_normalised=P.get("energy_normalised", False),
    )
    optimizer = {
        "amplitude": (torch.optim.SGD, dict(lr=P["lr_sgd"], momentum=P["amp_momentum"])),
        "shape": (torch.optim.Adam, dict(lr=P["lr_adam"])),
    }
    event = restore(
        "s01_ldi", C=C, H=HSet().zero(**P["weights"]), velocity=velocity,
        n_steps=P["n_steps"], integrator=P["integrator"], depth_axis=1,
        init_std=P["init_std"], sigma_floor=0.0, signed_normals=True,
        learning_rate=P["lr_adam"], optimizer=optimizer, pin=anchor,
    )
    for name, h in zip(reversed(horizon_names), reversed(horizons)):
        event.addIsosurface(name, seed=h)
    return curlew.GeoModel([event]), event


def octave_shares(event):
    # Each mode's share of the map's rms displacement, summed per wavelength bin.
    _, wavelength, share = diagnostics.displacement_spectrum(event, spectrum_points)[0]
    idx = np.digitize(wavelength / WIDTH, OCTAVES) - 1
    out = np.zeros(len(OCTAVES) - 1)
    np.add.at(out, np.clip(idx, 0, len(out) - 1), share)
    return out


def fit(band, seed, member):
    torch.manual_seed(404 + member)
    model, event = make_ldi(band, seed)
    model.prefit(epochs=P["epochs_ldi"], vb=False, early_stop=None)
    loss = float(event.loss().total().detach().cpu())
    d = diagnostics.evaluate_ensemble([model], [event], grd, G.shape, domaxes[0], domaxes[1], grid=G)
    units = recumbent_inputs.match_units(d["lithology"][0], unit_true)
    s = recumbent_inputs.score_against_truth(units, domaxes[0], domaxes[1], unit_true, reference)
    contacts = diagnostics.ensemble_contacts(d["scalar"], domaxes[0], domaxes[1], horizons)[0]
    return dict(loss=loss, extrema=int(d["extrema"].sum()), contact_distance=np.asarray(s["contact_distance"]),
                iou=np.asarray(s["iou"]), octaves=octave_shares(event), contacts=contacts)

## Seeds

The first `N_SEEDS` seeds of `01d`'s ensemble, so the fits at the main-text
band are members of it.

In [ ]:
N_SEEDS = 5
seed_rng = np.random.default_rng(P["fourier_seed"] + 91)
SEEDS = [int(P["fourier_seed"])]
while len(SEEDS) < N_SEEDS:
    candidate = int(seed_rng.integers(0, 2**31 - 1))
    if candidate not in SEEDS:
        SEEDS.append(candidate)
print(SEEDS)


def sweep(bands, label):
    # Returns stacked results, bands x seeds.
    rows = []
    t0 = perf_counter()
    for band in bands:
        row = []
        for member, seed in enumerate(SEEDS, start=1):
            r = fit(band, seed, member)
            row.append(r)
            print(f"{label} {band[0]:.0f}-{band[1]:.0f} m, seed {seed}: loss {r['loss']:.3g}, "
                  f"extrema {r['extrema']}, contact distance {np.round(r['contact_distance'], 1)} m, "
                  f"IoU {np.round(r['iou'], 2)}")
        rows.append(row)
    print(f"{label}: {perf_counter() - t0:.0f} s")
    return rows


def stack(rows, key):
    return np.array([[r[key] for r in row] for row in rows])


def contacts_of(rows):
    flat = [r["contacts"] for row in rows for r in row]
    return diagnostics.pack_contacts(flat)

## Band sweep

$L_{\min}$ and $L_{\max}$ on a grid around the main-text band (1–5 km, for a
1.2 km section), from wavelengths shorter than the fold to wavelengths ten
times the section.

In [ ]:
L_MIN = (250.0, 500.0, 1000.0, 2000.0)
L_MAX = (2000.0, 5000.0, 10000.0)
BANDS = [(lo, hi) for lo in L_MIN for hi in L_MAX if lo < hi]
band_rows = sweep(BANDS, "band")

In [ ]:
np.savez(
    RES_DIR / "band_sweep.npz",
    bands=np.array(BANDS), seeds=np.array(SEEDS), octaves=OCTAVES, width=WIDTH,
    loss=stack(band_rows, "loss"), extrema=stack(band_rows, "extrema"),
    contact_distance=stack(band_rows, "contact_distance"), iou=stack(band_rows, "iou"),
    octave_share=stack(band_rows, "octaves"), window=np.array(reference["window"]),
    **contacts_of(band_rows),
)
print("wrote", RES_DIR / "band_sweep.npz")

## Summary

In [ ]:
for label, rows, keys in (("band", band_rows, [f"{lo:.0f}-{hi:.0f} m" for lo, hi in BANDS]),):
    cd, iou, ext = stack(rows, "contact_distance"), stack(rows, "iou"), stack(rows, "extrema")
    print(label)
    for k, key in enumerate(keys):
        print(f"  {key:>14}: contact distance median {np.round(np.median(cd[k], axis=0), 1)} m, "
              f"IoU median {np.round(np.median(iou[k], axis=0), 2)}, extrema max {ext[k].max()}")